<a href="https://colab.research.google.com/github/Nishank018/ai-ml-practice/blob/main/Fake_news_detection_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Load LIAR Dataset from TSV and Assign Column Names

In this step, we will download the LIAR dataset (if not already present), load it into a pandas DataFrame, and then assign specific column names to make the data more interpretable and easier to work with for our fake news detection task.

In [16]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("doanquanvietnamca/liar-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'liar-dataset' dataset.
Path to dataset files: /kaggle/input/liar-dataset


In [17]:
import pandas as pd
import os

# Define column names based on LIAR dataset documentation
column_names = [
    'id', 'label', 'statement', 'subject', 'speaker', 'speaker_title',
    'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts',
    'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context'
]

# Construct the full path to the train.tsv file using the 'path' obtained from kagglehub
train_file_path = os.path.join(path, 'train.tsv')

# Load the TSV file into a pandas DataFrame
df = pd.read_csv(train_file_path, sep='\t', names=column_names)

# Display the first 5 rows and the DataFrame info to verify loading and columns
print("First 5 rows of the dataset:")
display(df.head())
print("\nDataFrame Info:")
df.info()

First 5 rows of the dataset:


,id,label,statement,subject,speaker,speaker_title,state_info,party_affiliation,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release
4,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10240 entries, 0 to 10239
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    10240 non-null  object 
 1   label                 10240 non-null  object 
 2   statement             10240 non-null  object 
 3   subject               10238 non-null  object 
 4   speaker               10238 non-null  object 
 5   speaker_title         7342 non-null   object 
 6   state_info            8030 non-null   object 
 7   party_affiliation     10238 non-null  object 
 8   barely_true_counts    10238 non-null  float64
 9   false_counts          10238 non-null  float64
 10  half_true_counts      10238 non-null  float64
 11  mostly_true_counts    10238 non-null  float64
 12  pants_on_fire_counts  10238 non-null  float64
 13  context               10138 non-null  object 
dtypes: float64(5), object(9)
memory usage: 1.1+ MB


### Step 2: Create 'content' column

In this step, we will create a new column named 'content' by directly copying the text from the 'statement' column. This simplifies future text processing by centralizing the primary text for analysis.

In [18]:
# Create the 'content' column by copying the 'statement' column
df['content'] = df['statement']

# Display the first few rows with the new 'content' column to verify
print("DataFrame with new 'content' column:")
display(df[['statement', 'content']].head())

# Verify that the 'content' column has been created
print("\nDataFrame Info after creating 'content' column:")
df.info()

DataFrame with new 'content' column:


,statement,content
0,Says the Annies List political group supports ...,Says the Annies List political group supports ...
1,When did the decline of coal start? It started...,When did the decline of coal start? It started...
2,"Hillary Clinton agrees with John McCain ""by vo...","Hillary Clinton agrees with John McCain ""by vo..."
3,Health care reform legislation is likely to ma...,Health care reform legislation is likely to ma...
4,The economic turnaround started at the end of ...,The economic turnaround started at the end of ...



DataFrame Info after creating 'content' column:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10240 entries, 0 to 10239
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    10240 non-null  object 
 1   label                 10240 non-null  object 
 2   statement             10240 non-null  object 
 3   subject               10238 non-null  object 
 4   speaker               10238 non-null  object 
 5   speaker_title         7342 non-null   object 
 6   state_info            8030 non-null   object 
 7   party_affiliation     10238 non-null  object 
 8   barely_true_counts    10238 non-null  float64
 9   false_counts          10238 non-null  float64
 10  half_true_counts      10238 non-null  float64
 11  mostly_true_counts    10238 non-null  float64
 12  pants_on_fire_counts  10238 non-null  float64
 13  context               10138 non-null  object 
 14  content              

### Step 3: Convert Labels to Binary

In this step, we will transform the original multi-class `label` column into a binary format. Labels like 'true' and 'mostly-true' will be mapped to `1` (indicating a 'real' statement), while 'false', 'barely-true', and 'pants-fire' will be mapped to `0` (indicating a 'fake' statement). Rows with the 'half-true' label will be removed from the dataset, as they are not explicitly classified as either real or fake for this binary task.

In [19]:
# Define the mapping for binary labels
label_mapping = {
    'true': 1,
    'mostly-true': 1,
    'false': 0,
    'barely-true': 0,
    'pants-fire': 0
}

# Apply the mapping to create a new 'binary_label' column
df['binary_label'] = df['label'].map(label_mapping)

# Remove rows where 'label' was 'half-true' (which would result in NaN in 'binary_label')
df.dropna(subset=['binary_label'], inplace=True)

# Convert 'binary_label' to integer type
df['binary_label'] = df['binary_label'].astype(int)

# Display the value counts of the new binary label and the first few rows
print("Value counts for the new 'binary_label' column:")
print(df['binary_label'].value_counts())

print("\nFirst 5 rows with 'label' and 'binary_label':")
display(df[['label', 'binary_label', 'statement']].head())

print("\nDataFrame Info after label conversion:")
df.info()

Value counts for the new 'binary_label' column:
binary_label
0    4488
1    3638
Name: count, dtype: int64

First 5 rows with 'label' and 'binary_label':


,label,binary_label,statement
0,false,0,Says the Annies List political group supports ...
2,mostly-true,1,"Hillary Clinton agrees with John McCain ""by vo..."
3,false,0,Health care reform legislation is likely to ma...
5,true,1,The Chicago Bears have had more starting quart...
6,barely-true,0,Jim Dunnam has not lived in the district he re...



DataFrame Info after label conversion:
<class 'pandas.core.frame.DataFrame'>
Index: 8126 entries, 0 to 10239
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    8126 non-null   object 
 1   label                 8126 non-null   object 
 2   statement             8126 non-null   object 
 3   subject               8124 non-null   object 
 4   speaker               8124 non-null   object 
 5   speaker_title         5818 non-null   object 
 6   state_info            6356 non-null   object 
 7   party_affiliation     8124 non-null   object 
 8   barely_true_counts    8124 non-null   float64
 9   false_counts          8124 non-null   float64
 10  half_true_counts      8124 non-null   float64
 11  mostly_true_counts    8124 non-null   float64
 12  pants_on_fire_counts  8124 non-null   float64
 13  context               8041 non-null   object 
 14  content               8126 non-null 

### Step 4: Clean Text

In this step, we will preprocess the text in the 'content' column to prepare it for machine learning. This involves converting all text to lowercase, removing URLs, eliminating special characters (keeping only alphanumeric characters and spaces), and finally, removing any extra whitespace. These cleaning steps help standardize the text and reduce noise, which is crucial for improving model performance.

In [20]:
import re

# Function to clean text
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower() # Lowercase
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Remove URLs
    text = re.sub(r'[^a-z0-9\s]', '', text) # Remove special characters, keep alphanumeric and spaces
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    return text

# Apply the cleaning function to the 'content' column
df['cleaned_content'] = df['content'].apply(clean_text)

# Display the first few rows with original and cleaned content to verify
print("First 5 rows with original and cleaned content:")
display(df[['content', 'cleaned_content']].head())

# Display some descriptive statistics for the new column
print("\nDescriptive statistics for 'cleaned_content' length:")
df['cleaned_content'].str.len().describe()

First 5 rows with original and cleaned content:


,content,cleaned_content
0,Says the Annies List political group supports ...,says the annies list political group supports ...
2,"Hillary Clinton agrees with John McCain ""by vo...",hillary clinton agrees with john mccain by vot...
3,Health care reform legislation is likely to ma...,health care reform legislation is likely to ma...
5,The Chicago Bears have had more starting quart...,the chicago bears have had more starting quart...
6,Jim Dunnam has not lived in the district he re...,jim dunnam has not lived in the district he re...



Descriptive statistics for 'cleaned_content' length:


,cleaned_content
count,8126.000000
mean,102.926778
std,60.954407
min,10.000000
25%,70.000000
50%,95.000000
75%,127.000000
max,3067.000000


### Step 7: Train and Evaluate Machine Learning Models (Logistic Regression and Linear SVM)

In this step, we will train two popular machine learning models for text classification: Logistic Regression and Linear Support Vector Machine (SVM). We will use the TF-IDF vectorized data (`X_train_tfidf` and `X_test_tfidf`) and the binary labels (`y_train` and `y_test`). After training each model, we will evaluate its performance using common classification metrics like accuracy, precision, recall, and F1-score.

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# --- TF-IDF Vectorization (Moved from Step 6 to resolve NameError) ---
print("\n--- Performing TF-IDF Vectorization ---")
# Initialize TF-IDF Vectorizer with specified parameters
tfidf_vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

# Fit the vectorizer on the training data and transform both training and testing data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Display the shape of the resulting TF-IDF matrices
print("Shape of X_train_tfidf:", X_train_tfidf.shape)
print("Shape of X_test_tfidf:", X_test_tfidf.shape)

# Optionally, you can print a few feature names to inspect
print("\nFirst 10 feature names:", tfidf_vectorizer.get_feature_names_out()[:10])


# --- Logistic Regression Model ---
print("\n--- Training Logistic Regression Model ---")
log_reg_model = LogisticRegression(max_iter=1000, random_state=42)
log_reg_model.fit(X_train_tfidf, y_train)

# Predict on the test set
y_pred_lr = log_reg_model.predict(X_test_tfidf)

# Evaluate Logistic Regression
print("Logistic Regression Metrics:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_lr):.4f}")

# --- Linear SVM Model ---
print("\n--- Training Linear SVM Model ---")
linear_svc_model = LinearSVC(random_state=42)
linear_svc_model.fit(X_train_tfidf, y_train)

# Predict on the test set
y_pred_svc = linear_svc_model.predict(X_test_tfidf)

# Evaluate Linear SVM
print("\nLinear SVM Metrics:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_svc):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_svc):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_svc):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_svc):.4f}")


--- Performing TF-IDF Vectorization ---
Shape of X_train_tfidf: (6500, 10000)
Shape of X_test_tfidf: (1626, 10000)

First 10 feature names: ['10' '10 billion' '10 million' '10 of' '10 percent' '10 times' '10 years'
 '100' '100 billion' '100 days']

--- Training Logistic Regression Model ---
Logistic Regression Metrics:
Accuracy: 0.6390
Precision: 0.6279
Recall: 0.4753
F1-Score: 0.5410

--- Training Linear SVM Model ---

Linear SVM Metrics:
Accuracy: 0.6089
Precision: 0.5695
Recall: 0.5179
F1-Score: 0.5424


### Step 8: Build and Evaluate BERT Model

In this crucial step, we will leverage a pre-trained BERT model for our fake news detection task. We will fine-tune a BERT-based classifier on our LIAR dataset. After training, we will evaluate its performance using accuracy, precision, recall, and F1-score to compare with the traditional machine learning models and assess the impact of transfer learning.

In [22]:
!pip install transformers datasets accelerate -qq


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import numpy as np
import torch

# Load pre-trained BERT tokenizer and model
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Prepare the dataset for BERT
# Create a Hugging Face Dataset from your pandas DataFrame
hf_dataset = Dataset.from_pandas(df[['cleaned_content', 'binary_label']])

# Tokenization function
def tokenize_function(examples):
    return tokenizer(examples["cleaned_content"], truncation=True, padding="max_length", max_length=128)

# Apply tokenization to the dataset
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)

# Rename 'binary_label' to 'labels' and set format for PyTorch
tokenized_dataset = tokenized_dataset.rename_column("binary_label", "labels")
tokenized_dataset = tokenized_dataset.remove_columns(["cleaned_content", "__index_level_0__"])
tokenized_dataset.set_format("torch")

# Split the tokenized dataset into train and test sets for BERT
# Use the same split ratio as before (80/20)
bert_train_dataset, bert_test_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)['train'], tokenized_dataset.train_test_split(test_size=0.2, seed=42)['test']


# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,              # total number of training epochs
    per_device_train_batch_size=16,  # batch size per device during training
    per_device_eval_batch_size=16,   # batch size per device during evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir="./logs",            # directory for storing logs
    logging_steps=10,
    report_to="none"                 # disable reporting to services like W&B
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Define compute_metrics function
def compute_metrics(p):
    predictions = np.argmax(p.predictions, axis=1)
    accuracy = accuracy_score(p.label_ids, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(p.label_ids, predictions, average='binary')
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=bert_train_dataset,
    eval_dataset=bert_test_dataset,
    compute_metrics=compute_metrics
)

# Train the model
print("\n--- Fine-tuning BERT Model ---")
trainer.train()

# Evaluate the model
print("\n--- Evaluating BERT Model ---")
eval_results = trainer.evaluate()
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/8126 [00:00<?, ? examples/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



--- Fine-tuning BERT Model ---


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


### Step 9: Create a Prediction Function

In this step, we will create a function that takes new text data as input and uses our best-performing model to predict whether the statement is fake (0) or real (1). This function will incorporate the same preprocessing steps (text cleaning and TF-IDF vectorization/tokenization) used during training to ensure consistency.

In [ ]:
def predict_fake_news(text, model, vectorizer=None, tokenizer=None, device='cpu'):
    """
    Predicts whether a given text statement is fake (0) or real (1) using a trained model.

    Args:
        text (str): The input text statement.
        model: The trained machine learning model (Logistic Regression, Linear SVM, or BERT).
        vectorizer: Fitted TF-IDF vectorizer if using traditional ML models.
        tokenizer: Fitted BERT tokenizer if using a BERT model.
        device (str): The device to run BERT model on ('cpu' or 'cuda').

    Returns:
        int: The predicted label (0 for fake, 1 for real).
    """
    # 1. Clean the input text
    cleaned_text = clean_text(text)

    if isinstance(model, LogisticRegression) or isinstance(model, LinearSVC):
        # 2. Vectorize the cleaned text using the fitted TF-IDF vectorizer
        if vectorizer is None:
            raise ValueError("TF-IDF vectorizer must be provided for traditional ML models.")
        text_vectorized = vectorizer.transform([cleaned_text])

        # 3. Make prediction
        prediction = model.predict(text_vectorized)[0]

    elif hasattr(model, 'bert'): # Simple check for a HuggingFace BERT-like model
        if tokenizer is None:
            raise ValueError("Tokenizer must be provided for BERT model.")

        # 2. Tokenize the cleaned text
        inputs = tokenizer(cleaned_text, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # 3. Make prediction
        model.to(device)
        model.eval() # Set model to evaluation mode
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            prediction = torch.argmax(logits, dim=1).item()
    else:
        raise TypeError("Unsupported model type. Please provide a LogisticRegression, LinearSVC, or BERT model.")

    return prediction

# Example Usage with Logistic Regression
print("\n--- Testing Prediction Function with Logistic Regression ---")
sample_text_lr_true = "The president stated that the economy grew by 5% last quarter, which is a fact confirmed by government reports."
sample_text_lr_fake = "Aliens landed in Washington D.C. last night and met with top officials, according to a secret source."

pred_lr_true = predict_fake_news(sample_text_lr_true, log_reg_model, vectorizer=tfidf_vectorizer)
pred_lr_fake = predict_fake_news(sample_text_lr_fake, log_reg_model, vectorizer=tfidf_vectorizer)

print(f"'{{sample_text_lr_true}}' -> Predicted: {'Real' if pred_lr_true == 1 else 'Fake'}")
print(f"'{{sample_text_lr_fake}}' -> Predicted: {'Real' if pred_lr_fake == 1 else 'Fake'}")

# Example Usage with BERT (assuming 'model' and 'tokenizer' from BERT step are available)
# Note: This will run on CPU if 'cuda' is not available or not specified
print("\n--- Testing Prediction Function with BERT ---")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

pred_bert_true = predict_fake_news(sample_text_lr_true, model, tokenizer=tokenizer, device=device)
pred_bert_fake = predict_fake_news(sample_text_lr_fake, model, tokenizer=tokenizer, device=device)

print(f"'{{sample_text_lr_true}}' -> Predicted: {'Real' if pred_bert_true == 1 else 'Fake'}")
print(f"'{{sample_text_lr_fake}}' -> Predicted: {'Real' if pred_bert_fake == 1 else 'Fake'}")

In [28]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = df['cleaned_content']
y = df['binary_label']

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Display the shapes of the resulting datasets to verify the split
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

# Display the distribution of labels in training and test sets to ensure stratification worked
print("\nDistribution of labels in y_train:")
print(y_train.value_counts(normalize=True))
print("\nDistribution of labels in y_test:")
print(y_test.value_counts(normalize=True))

Shape of X_train: (6500,)
Shape of X_test: (1626,)
Shape of y_train: (6500,)
Shape of y_test: (1626,)

Distribution of labels in y_train:
binary_label
0    0.552308
1    0.447692
Name: proportion, dtype: float64

Distribution of labels in y_test:
binary_label
0    0.552276
1    0.447724
Name: proportion, dtype: float64
